In [13]:
from linearmodels.iv import IV2SLS
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS

Data: $(Y_{it}, X_{i}, Z_i)$ where 
* $Y_{it}$ is sales outcomes of the book
* $X_{i}$ is Books3 status  
* $Z_{i}$ is shares (serves as IV)
* and $t$ is on a month-year level

Outcome variable of interest is $\Delta Y_{i}$ where the regression would be run on $(\Delta Y_{i}, X_i, Z_i)$
* Construction of this outcome variable is itself not trivial

Potential ways to measure $\Delta Y_{i}$:
* log(average sales rank after ChatGPT) - log(average sales rank before ChatGPT)
  * taking the average might be losing useful information about the variance over time
* $log(Y_{i,t_1}) - log(Y_{i, t_0})$
  * Might be really noisy b/c some books data looks quite cyclical
    
Exclusion restriction issues now that outcome is sales measure (not a huge issue with reviewers):
* $Z_i$ is constructed purely off of Books3 distribution
  * Does the fact that there are a lot of books in mid 2000s impact the change in sales rank
  * ...
  
Or what if we just try to work with all the data $Y_{it}$ and avoid aggregations that lose information!
* Let $W_{it} = X_{i} \times \text{Post}_t$
* Then $g(Y_{it}) = \alpha_i + \lambda_t + \beta W_{it} + u_{it}$

Due to potential unobserved confounders we now consider IV:
* First stage: $W_{it} = \alpha_i + \lambda_t + \pi(Z_i \times \text{Post}_t) + v_{it}$
* Second stage: $Y_{it} = \alpha_i + \lambda_t + \beta(\hat W_{it}) + \epsilon_{it}$

Before whereas we needed to define a way to measure $\Delta$ now with time and unit FE, the "richness" of the time series data is all incorporated...?

Brainstorm on: https://chatgpt.com/share/e/6a04cae2-4604-8008-aa4c-71caf3183715

In [3]:
df = pd.read_csv('/Users/stjia/Documents/research/cloze-encounters/keepa-analysis/data/keepa-merged-long.csv')

## Create Outcome Variables

In [9]:
df.head()

,book_id,isbn13,date,price,sales_rank,treated,num_ratings,num_reviews,pub_year,shares,popularity,Post
0,0,9780521011488,2019-01-01,9.62,519883.0,0,75.0,4.0,2003.0,0.005479,10-100,0
1,0,9780521011488,2019-02-01,9.79,534904.0,0,75.0,4.0,2003.0,0.005479,10-100,0
2,0,9780521011488,2019-03-01,9.91,584180.0,0,75.0,4.0,2003.0,0.005479,10-100,0
3,0,9780521011488,2019-04-01,10.28,466902.0,0,75.0,4.0,2003.0,0.005479,10-100,0
4,0,9780521011488,2019-05-01,9.60,473655.0,0,75.0,4.0,2003.0,0.005479,10-100,0


In [8]:
# add a Post variable
CUTOFF = pd.Timestamp('2022-11-01')
df['date'] = pd.to_datetime(df['date'])
df['Post'] = (df['date'] >= CUTOFF).astype(int)

# change treated to binary int
df['treated'] = df['treated'].astype(int)

In [11]:
# summary statistics for df
print(df.describe())

             book_id        isbn13                 date          price  \
count  978966.000000  9.789660e+05               978966  904019.000000   
mean     6203.760301  9.780958e+12  2022-05-01 22:13:20       9.504077   
min         0.000000  9.780000e+12  2019-01-01 00:00:00       0.000000   
25%      3063.000000  9.780333e+12  2020-09-01 00:00:00       8.840000   
50%      6122.500000  9.780746e+12  2022-05-01 00:00:00       9.510000   
75%      9186.000000  9.781409e+12  2024-01-01 00:00:00      10.010000   
max     13129.000000  9.798988e+12  2025-09-01 00:00:00  107374.190000   
std      3666.076552  1.233969e+09                  NaN     195.658344   

         sales_rank        treated   num_ratings    num_reviews  \
count  9.706460e+05  978966.000000  9.789660e+05  978966.000000   
mean   1.511025e+06       0.487175  1.388343e+04     627.160434   
min    3.310000e+02       0.000000  0.000000e+00       0.000000   
25%    4.799160e+05       0.000000  1.200000e+01       1.000000  

## IV on $\Delta Y_{it}$

## TWFE-IV on $Y_{it}$

In [ ]:
df["W"] = df["treated"] * df["Post"] # treatment x post
df["Z_post"] = df["shares"] * df["Post"] # IV x post

In [15]:
# no IV
df = df.set_index(["book_id", "date"]) # prep for unit and time FE
model = PanelOLS.from_formula(
    "sales_rank ~ 1 + W + EntityEffects + TimeEffects",
    data=df
)

res = model.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)


KeyError: "None of ['book_id', 'date'] are in the columns"

In [ ]:

model = IV2SLS.from_formula(
    "y ~ 1 + controls + [endog_var ~ instrument]",
    data=df
)

results = model.fit()
print(results.summary)